In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
#combining all the txt files into a combined data file (take neutral from this dataset)

file_list = ['../raw_data/file1.txt', '../raw_data/file2.txt', '../raw_data/file3.txt', '../raw_data/file4.txt', '../raw_data/file5.txt' ]

with open('../raw_data/combined_semeval.txt', 'w', encoding='utf-8') as outfile:
    for fname in file_list:
        with open(fname, 'r', encoding='utf-8') as infile:
            outfile.write(infile.read())
            outfile.write('\n')

In [ ]:
df = pd.read_csv('../raw_data/combined_semeval.txt',sep='\t')

In [ ]:
df.columns = ['id', 'sentiment', 'text']

In [ ]:
#dropping the id column to keep just 'text' and 'sentiment'

df = df.drop('id', axis=1)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
#keeping just the neutral values

neutral_df = df[df['sentiment'] == 'neutral']

In [ ]:
neutral_df['sentiment'] = neutral_df['sentiment'].map({
    "negative": 0,
    "positive": 1,
    "neutral": 2
})

In [ ]:
twitter_data = pd.read_csv('../raw_data/twitter_dataset.csv')

In [ ]:
twitter_data.head()

In [ ]:
twitter_data = twitter_data.drop(columns=['Unnamed: 0'])

In [ ]:
twitter_data.columns = ['text','sentiment']

In [ ]:
df_combined = pd.concat([twitter_data, neutral_df], ignore_index=True)

In [ ]:
df_combined

In [ ]:
df_combined.info

In [ ]:
#82 values initially null form twitter_dataset

df_combined.isnull().sum()

In [ ]:
# dropping rows with null values

df_combined = df_combined.dropna(subset=['text', 'sentiment'])

In [ ]:
#simple text clean

df_combined['text'] = df_combined['text'].astype(str).str.strip()
df_combined = df_combined[df_combined['text'] != ""]

In [ ]:
target_n = 35000

df_balanced = (
    df_combined.groupby('sentiment', group_keys=False)
    .sample(n=target_n, random_state=42, replace=False)
)

In [ ]:
df_balanced.head()

In [ ]:
df_balanced['sentiment'].value_counts()

In [ ]:
import re

df_balanced.isnull().sum()

CLEANING

In [ ]:
def clean_text(text):
    text = text.lower()
    
    # fix common encoding issues
    text = text.replace("Â´", "'")
    
    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # remove mentions (@user)
    text = re.sub(r"@\w+", "", text)
    
    # remove hashtags symbol ONLY (#happy → happy)
    text = re.sub(r"#", "", text)
    
    return text

In [ ]:
def normalize_repeats(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

In [ ]:
def remove_special_chars(text):
    text = re.sub(r"[^a-zA-Z_\s]", "", text)
    return text

In [ ]:
def preprocess_pipeline(text):
    text = clean_text(text)
    text = normalize_repeats(text)
    text = remove_special_chars(text)
    return text

df['cleaned_text'] = df['text'].apply(preprocess_pipeline)

In [ ]:
df.to_csv('../clean_data/re_clean_dataset.csv')